In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix)
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.linear_model import RidgeClassifier

In [8]:
def load_fasta_files(file_paths, predefined_labels):
    sequences = []
    labels = []
    for file_path, label in zip(file_paths, predefined_labels):
        with open(file_path, 'r') as file:
            seq = ''
            for line in file:
                if line.startswith('>'):
                    if seq:
                        sequences.append(seq)
                        labels.append(label)
                    seq = ''
                else:
                    seq += line.strip()
            if seq:
                sequences.append(seq)
                labels.append(label)
    return sequences, labels

# Simple k-mer counting for preprocessing
def preprocess_sequences(sequences, k=3):
    kmer_vectorizer = CountVectorizer(analyzer='char', ngram_range=(k, k))
    X = kmer_vectorizer.fit_transform(sequences)
    X = X [:, :1000]
    return X

# Evaluation process using 4-fold cross-validation and final test set
def evaluate_model(X_train, y_train, X_test, y_test, model, num_classes):
    kf = KFold(n_splits=4, shuffle=True, random_state=42)
    fold_results = {
        'Accuracy': [], 'Precision': [], 'Recall': [], 'F1-Score': [], 'MCC': [],
        'AUROC': [], 'AUPRC': [], 'Confusion Matrix': [], 'Specificity': []
    }
    all_y_true = []
    all_y_pred = []
    all_y_pred_probs = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"Starting fold {fold + 1}...")

        X_fold_train, X_val = X_train[train_idx], X_train[val_idx]
        y_fold_train, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_val)
        y_val_pred_probs = model.predict_proba(X_val)

        # Standard metrics
        fold_results['Accuracy'].append(accuracy_score(y_val, y_val_pred))
        fold_results['Precision'].append(precision_score(y_val, y_val_pred, average='weighted', zero_division=0))
        fold_results['Recall'].append(recall_score(y_val, y_val_pred, average='weighted', zero_division=0))
        fold_results['F1-Score'].append(f1_score(y_val, y_val_pred, average='weighted', zero_division=0))
        fold_results['MCC'].append(matthews_corrcoef(y_val, y_val_pred))
        
        # Confusion matrix
        cm = confusion_matrix(y_val, y_val_pred)
        
        # Calculate specificity for each class
        specificity = []
        for i in range(num_classes):
            tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]  # True Negatives
            fp = cm[:, i].sum() - cm[i, i]  # False Positives
            specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0)  # Avoid division by zero
        
        # Average specificity across classes
        fold_results['Specificity'].append(sum(specificity) / num_classes)

        # For AUROC and AUPRC, convert the predictions and true values to one-hot encoded format
        y_val_onehot = to_categorical(y_val, num_classes=num_classes)
        fold_results['AUROC'].append(roc_auc_score(y_val_onehot, y_val_pred_probs, multi_class='ovr'))
        fold_results['AUPRC'].append(average_precision_score(y_val_onehot, y_val_pred_probs, average='macro'))

        # Store results for final evaluation
        all_y_true.extend(y_val)
        all_y_pred.extend(y_val_pred)
        all_y_pred_probs.extend(y_val_pred_probs)

        print(f"Fold {fold + 1} done.")

    # Final test set evaluation
    y_test_pred = model.predict(X_test)
    y_test_pred_probs = model.predict_proba(X_test)
    
    # Calculate specificity for the test set
    cm_test = confusion_matrix(y_test, y_test_pred)
    test_specificity = []
    for i in range(num_classes):
        tn = cm_test.sum() - cm_test[i, :].sum() - cm_test[:, i].sum() + cm_test[i, i]
        fp = cm_test[:, i].sum() - cm_test[i, i]
        test_specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

    final_test_specificity = sum(test_specificity) / num_classes

    test_results = {
        'Accuracy': accuracy_score(y_test, y_test_pred),
        'Precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_test_pred),
        'AUROC': roc_auc_score(to_categorical(y_test, num_classes=num_classes), y_test_pred_probs, multi_class='ovr'),
        'AUPRC': average_precision_score(to_categorical(y_test, num_classes=num_classes), y_test_pred_probs, average='macro'),
        'Specificity': final_test_specificity
    }


    return fold_results, test_results

def print_results(fold_results, test_results, num_folds=4):
    """
    Prints fold-wise and final test set results.

    Parameters:
    - fold_results: Dictionary containing evaluation metrics for each fold.
    - test_results: Dictionary containing evaluation metrics for the final test set.
    - num_folds: Number of cross-validation folds (default is 4).
    """
    print("\nFold-wise Results:")
    for i in range(num_folds):
        print(f"Fold {i + 1}: Accuracy={fold_results['Accuracy'][i]:.4f}, "
              f"Precision={fold_results['Precision'][i]:.4f}, "
              f"Recall={fold_results['Recall'][i]:.4f}, "
              f"F1={fold_results['F1-Score'][i]:.4f}, "
              f"MCC={fold_results['MCC'][i]:.4f}, "
              f"AUROC={fold_results['AUROC'][i]:.4f}, "
              f"AUPRC={fold_results['AUPRC'][i]:.4f}, "
              f"Specificity={fold_results['Specificity'][i]:.4f}")
    
    print("\nFinal Test Set Results:")
    print(f"Accuracy={test_results['Accuracy']:.4f}, "
      f"Precision={test_results['Precision']:.4f}, "
      f"Recall={test_results['Recall']:.4f}, "
      f"F1={test_results['F1-Score']:.4f}, "
      f"MCC={test_results['MCC']:.4f}, "
      f"AUROC={test_results['AUROC']:.4f}, "
      f"AUPRC={test_results['AUPRC']:.4f}, "
      f"Specificity={test_results['Specificity']:.4f}")

# Main code
fasta_files = ['covid.fasta', 'dengue.fasta', 'hepatitis.fasta', 'influenza.fasta', 'mers.fasta']
predefined_labels = ['COVID', 'Dengue', 'Hepatitis', 'Influenza', 'MERS']

# Load data
sequences, labels = load_fasta_files(fasta_files, predefined_labels)

In [3]:
# Preprocess sequences
X = preprocess_sequences(sequences, k=16)

# Encode labels into numeric form
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
num_classes = len(np.unique(y))

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Initialize Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=5, 
    max_depth=3,     
    min_samples_split=20,   
    min_samples_leaf=10,   
    max_features='sqrt',   
    random_state=42
)

# Evaluate the model
rf_fold_results, rf_test_results = evaluate_model(X_train, y_train, X_test, y_test, rf_model, num_classes)
print_results(rf_fold_results, rf_test_results)

Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.5842, Precision=0.5212, Recall=0.5842, F1=0.4398, MCC=0.1828, AUROC=0.6432, AUPRC=0.3862, Specificity=0.8080
Fold 2: Accuracy=0.5862, Precision=0.5779, Recall=0.5862, F1=0.4421, MCC=0.1477, AUROC=0.6169, AUPRC=0.3770, Specificity=0.8054
Fold 3: Accuracy=0.5842, Precision=0.5698, Recall=0.5842, F1=0.4385, MCC=0.1201, AUROC=0.6429, AUPRC=0.3907, Specificity=0.8038
Fold 4: Accuracy=0.5875, Precision=0.3771, Recall=0.5875, F1=0.4385, MCC=0.0930, AUROC=0.6198, AUPRC=0.3519, Specificity=0.8021

Final Test Set Results:
Accuracy=0.5785, Precision=0.3709, Recall=0.5785, F1=0.4281, MCC=0.0972, AUROC=0.6265, AUPRC=0.3611, Specificity=0.8023


In [10]:
log_reg_model = LogisticRegression(
    max_iter=10,           
    solver='lbfgs'
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Evaluate the model
lr_fold_results, lr_test_results = evaluate_model(X_train, y_train, X_test, y_test, log_reg_model, num_classes)
print_results(lr_fold_results, lr_test_results)

Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.6440, Precision=0.6249, Recall=0.6440, F1=0.5441, MCC=0.3750, AUROC=0.7399, AUPRC=0.5651, Specificity=0.8356
Fold 2: Accuracy=0.6439, Precision=0.6279, Recall=0.6439, F1=0.5412, MCC=0.3594, AUROC=0.7503, AUPRC=0.5708, Specificity=0.8325
Fold 3: Accuracy=0.6327, Precision=0.6197, Recall=0.6327, F1=0.5213, MCC=0.3262, AUROC=0.7477, AUPRC=0.5640, Specificity=0.8267
Fold 4: Accuracy=0.6474, Precision=0.6271, Recall=0.6474, F1=0.5429, MCC=0.3511, AUROC=0.7474, AUPRC=0.5611, Specificity=0.8308

Final Test Set Results:
Accuracy=0.6492, Precision=0.6271, Recall=0.6492, F1=0.5479, MCC=0.3773, AUROC=0.7589, AUPRC=0.5740, Specificity=0.8355


In [11]:
knn_model = KNeighborsClassifier(n_neighbors=100, p=1)
X_train_reduced = X_train[:, :1000]  # Use only the first 10 features
X_test_reduced = X_test[:, :1000]

knn_fold_results, knn_test_results = evaluate_model(X_train, y_train, X_test, y_test, knn_model, num_classes)
print_results(knn_fold_results, knn_test_results)


Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.6615, Precision=0.7369, Recall=0.6615, F1=0.5699, MCC=0.4150, AUROC=0.7698, AUPRC=0.5764, Specificity=0.8442
Fold 2: Accuracy=0.6704, Precision=0.7786, Recall=0.6704, F1=0.5835, MCC=0.4222, AUROC=0.7672, AUPRC=0.5730, Specificity=0.8452
Fold 3: Accuracy=0.6696, Precision=0.7777, Recall=0.6696, F1=0.5806, MCC=0.4190, AUROC=0.7663, AUPRC=0.5728, Specificity=0.8443
Fold 4: Accuracy=0.6723, Precision=0.7666, Recall=0.6723, F1=0.5843, MCC=0.4113, AUROC=0.7651, AUPRC=0.5696, Specificity=0.8431

Final Test Set Results:
Accuracy=0.6720, Precision=0.7853, Recall=0.6720, F1=0.5859, MCC=0.4289, AUROC=0.7753, AUPRC=0.5835, Specificity=0.8464


In [12]:
nb_model = MultinomialNB()

nb_fold_results, nb_test_results = evaluate_model(X_train, y_train, X_test, y_test, nb_model, num_classes)
print_results(nb_fold_results, nb_test_results)

Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.6872, Precision=0.7442, Recall=0.6872, F1=0.6127, MCC=0.4671, AUROC=0.7722, AUPRC=0.5838, Specificity=0.8565
Fold 2: Accuracy=0.6933, Precision=0.7362, Recall=0.6933, F1=0.6179, MCC=0.4694, AUROC=0.7680, AUPRC=0.5773, Specificity=0.8567
Fold 3: Accuracy=0.6921, Precision=0.7313, Recall=0.6921, F1=0.6170, MCC=0.4642, AUROC=0.7681, AUPRC=0.5773, Specificity=0.8558
Fold 4: Accuracy=0.6948, Precision=0.7344, Recall=0.6948, F1=0.6188, MCC=0.4600, AUROC=0.7628, AUPRC=0.5708, Specificity=0.8546

Final Test Set Results:
Accuracy=0.6964, Precision=0.7419, Recall=0.6964, F1=0.6225, MCC=0.4785, AUROC=0.7747, AUPRC=0.5864, Specificity=0.8585


In [13]:
xgb_model = XGBClassifier(
    n_estimators=5, 
    max_depth=2, 
    learning_rate=0.3, 
    random_state=42
)

# Evaluate the decision tree model
xgb_fold_results, xgb_test_results = evaluate_model(X_train, y_train, X_test, y_test, xgb_model, num_classes)

# Print the results for each fold and the final test set
print_results(xgb_fold_results, xgb_test_results)


Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.6296, Precision=0.6195, Recall=0.6296, F1=0.5193, MCC=0.3401, AUROC=0.7343, AUPRC=0.5175, Specificity=0.8290
Fold 2: Accuracy=0.6409, Precision=0.6267, Recall=0.6409, F1=0.5324, MCC=0.3545, AUROC=0.7287, AUPRC=0.5107, Specificity=0.8311
Fold 3: Accuracy=0.6384, Precision=0.6218, Recall=0.6384, F1=0.5271, MCC=0.3455, AUROC=0.7245, AUPRC=0.5028, Specificity=0.8294
Fold 4: Accuracy=0.6459, Precision=0.6265, Recall=0.6459, F1=0.5371, MCC=0.3491, AUROC=0.7226, AUPRC=0.5001, Specificity=0.8300

Final Test Set Results:
Accuracy=0.6450, Precision=0.6255, Recall=0.6450, F1=0.5385, MCC=0.3683, AUROC=0.7344, AUPRC=0.5145, Specificity=0.8335


In [14]:
et_model = ExtraTreesClassifier(n_estimators=5, max_depth=3, random_state=42)


# Evaluate the decision tree model
et_fold_results, et_test_results = evaluate_model(X_train, y_train, X_test, y_test, et_model, num_classes)

# Print the results for each fold and the final test set
print_results(et_fold_results, et_test_results)

Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.5857, Precision=0.5739, Recall=0.5857, F1=0.4485, MCC=0.1801, AUROC=0.6232, AUPRC=0.3592, Specificity=0.8087
Fold 2: Accuracy=0.5916, Precision=0.5797, Recall=0.5916, F1=0.4545, MCC=0.1725, AUROC=0.6284, AUPRC=0.3709, Specificity=0.8080
Fold 3: Accuracy=0.5850, Precision=0.3692, Recall=0.5850, F1=0.4377, MCC=0.1335, AUROC=0.6327, AUPRC=0.3702, Specificity=0.8042
Fold 4: Accuracy=0.6002, Precision=0.5802, Recall=0.6002, F1=0.4611, MCC=0.1814, AUROC=0.6262, AUPRC=0.3635, Specificity=0.8081

Final Test Set Results:
Accuracy=0.5957, Precision=0.5750, Recall=0.5957, F1=0.4562, MCC=0.2058, AUROC=0.6305, AUPRC=0.3680, Specificity=0.8103


In [ ]:
ridge_model = RidgeClassifier(max_iter=50)  # Reduce max_iter for faster computation


# Evaluate the decision tree model
ridge_fold_results, ridge_test_results = evaluate_model(X_train, y_train, X_test, y_test, ridge_model, num_classes)

# Print the results for each fold and the final test set
print_results(ridge_fold_results, ridge_test_results)


In [15]:
gb_model = GradientBoostingClassifier(
    n_estimators=5,
    max_depth=3,
    learning_rate=0.1,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)



# Evaluate the decision tree model
gb_fold_results, gb_test_results = evaluate_model(X_train, y_train, X_test, y_test, gb_model, num_classes)

# Print the results for each fold and the final test set
print_results(gb_fold_results, gb_test_results)

Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.6349, Precision=0.6215, Recall=0.6349, F1=0.5244, MCC=0.3561, AUROC=0.7389, AUPRC=0.5305, Specificity=0.8314
Fold 2: Accuracy=0.6408, Precision=0.6267, Recall=0.6408, F1=0.5323, MCC=0.3542, AUROC=0.7300, AUPRC=0.5156, Specificity=0.8311
Fold 3: Accuracy=0.6384, Precision=0.6218, Recall=0.6384, F1=0.5271, MCC=0.3455, AUROC=0.7305, AUPRC=0.5164, Specificity=0.8294
Fold 4: Accuracy=0.6459, Precision=0.6265, Recall=0.6459, F1=0.5371, MCC=0.3491, AUROC=0.7224, AUPRC=0.5043, Specificity=0.8300

Final Test Set Results:
Accuracy=0.6450, Precision=0.6255, Recall=0.6450, F1=0.5385, MCC=0.3683, AUROC=0.7344, AUPRC=0.5183, Specificity=0.8335


In [17]:
lgbm_model = LGBMClassifier(
    n_estimators=5,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)



X_train_reduced = X_train_reduced.astype('float32')
X_test_reduced = X_test_reduced.astype('float32')

# Evaluate the decision tree model
lgbm_fold_results, lgbm_test_results = evaluate_model(X_train, y_train, X_test, y_test, lgbm_model, num_classes)

# Print the results for each fold and the final test set
print_results(lgbm_fold_results, lgbm_test_results)

Starting fold 1...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005827 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 191
[LightGBM] [Info] Number of data points in the train set: 32250, number of used features: 85
[LightGBM] [Info] Start training from score -0.548155
[LightGBM] [Info] Start training from score -3.324236
[LightGBM] [Info] Start training from score -1.872112
[LightGBM] [Info] Start training from score -1.600947
[LightGBM] [Info] Start training from score -3.490664
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

In [16]:
cb_model = CatBoostClassifier(
    iterations=5,
    depth=3,
    learning_rate=0.1,
    random_seed=42,
    verbose=False
)

# Evaluate the decision tree model
cb_fold_results, cb_test_results = evaluate_model(X_train, y_train, X_test, y_test, cb_model, num_classes)

# Print the results for each fold and the final test set
print_results(cb_fold_results, cb_test_results)


Starting fold 1...
Fold 1 done.
Starting fold 2...
Fold 2 done.
Starting fold 3...
Fold 3 done.
Starting fold 4...
Fold 4 done.

Fold-wise Results:
Fold 1: Accuracy=0.6092, Precision=0.6122, Recall=0.6092, F1=0.4800, MCC=0.2858, AUROC=0.6939, AUPRC=0.4667, Specificity=0.8195
Fold 2: Accuracy=0.6206, Precision=0.6192, Recall=0.6206, F1=0.4970, MCC=0.2986, AUROC=0.6844, AUPRC=0.4450, Specificity=0.8216
Fold 3: Accuracy=0.6207, Precision=0.6153, Recall=0.6207, F1=0.4958, MCC=0.2961, AUROC=0.6933, AUPRC=0.4611, Specificity=0.8211
Fold 4: Accuracy=0.6261, Precision=0.6191, Recall=0.6261, F1=0.5033, MCC=0.2920, AUROC=0.6846, AUPRC=0.4473, Specificity=0.8206

Final Test Set Results:
Accuracy=0.6226, Precision=0.6171, Recall=0.6226, F1=0.5001, MCC=0.3088, AUROC=0.6989, AUPRC=0.4663, Specificity=0.8230


In [4]:
def count_sequences_in_fasta(file_path):
    sequence_count = 0
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith(">"):
                sequence_count += 1
    return sequence_count

# Example usage
file_path = 'mers.fasta'
total_sequences = count_sequences_in_fasta(file_path)
print(f"Total number of DNA sequences: {total_sequences}")

Total number of DNA sequences: 1658
